In [1]:
### Importing the required library 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import MolStandardize
import joblib
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors,rdMolDescriptors


In [4]:
### Installation of the basic library 
from rdkit import Chem,DataStructs
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import Crippen
#import transformers
#from transformers import AutoModel, AutoTokenizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


In [5]:
### Reading the preprocess data from disk 
train_set=pd.read_csv('../data/final_data/final_unique_train.csv')

test_set=pd.read_csv('../data/final_data/final_unique_test.csv')

print(train_set.shape)
print(test_set.shape)

(17937, 8)
(1282, 8)


In [6]:
import torch
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdchem
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import DataLoader
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import pandas as pd
from rdkit import Chem
from torch_geometric.nn import MessagePassing
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import from_smiles

In [7]:
# Define a function to convert SMILES to PyTorch Geometric Data
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")

    # Get atom features
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append([atom.GetAtomicNum()])  # Example: Atomic number as feature
    x = torch.tensor(atom_features, dtype=torch.float)

    # Get bond features (edges)
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])  # Undirected graph

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index)

In [8]:
import utilities 

2025-04-19 00:58:44.111095: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [9]:
class MolecularDataset(torch.utils.data.Dataset):
    def __init__(self, graph_data_list, physical_properties, targets):
        self.graph_data_list = graph_data_list  # PyTorch Geometric Data objects
        self.physical_properties = torch.tensor(physical_properties, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.graph_data_list)

    def __getitem__(self, idx):
        return self.graph_data_list[idx], self.physical_properties[idx], self.targets[idx]

In [10]:
### Taking out the label data 
y_train=train_set['LogS']
y_test=test_set['LogS']

In [11]:
df123_train=utilities.generate123(train_set.smiles_canon)
df123_test=utilities.generate123(test_set.smiles_canon)


In [12]:
df38_train=utilities.generate_features38(train_set.smiles_canon)
df38_test=utilities.generate_features38(test_set.smiles_canon)

In [13]:
df7_train=utilities.get_functional_groups(train_set.smiles_canon)
df7_test=utilities.get_functional_groups(test_set.smiles_canon)

In [38]:
#df128_train=utilities.fingerprint(train_set.smiles_canon,2,128)
#df128_test=utilities.fingerprint(test_set.smiles_canon,2,128)

In [31]:
### Adding the features 
df163_train=pd.concat([df123_train, df38_train], axis=1)
df163_test=pd.concat([df123_test, df38_test], axis=1)
### Adding the functional group 
df170_train=pd.concat([df163_train, df7_train], axis=1)
df170_test=pd.concat([df163_test, df7_test], axis=1)

### Adding the functional group 
#df298_train=pd.concat([df170_train, df128_train], axis=1)
#df298_test=pd.concat([df170_test, df128_test], axis=1)


In [32]:
df_train2 = pd.concat([df170_train, y_train], axis=1)
df_test2 = pd.concat([df170_test, y_test], axis=1)
df_train = df_train2.replace((np.inf, -np.inf, np.nan), 0).reset_index(drop=True)
df_test = df_test2.replace((np.inf, -np.inf, np.nan), 0).reset_index(drop=True)
X_train = df_train.iloc[:, :-1]  # All columns except the last one
y_train = df_train.iloc[:, -1] 
X_test = df_test.iloc[:, :-1]  # All columns except the last one
y_test = df_test.iloc[:, -1] 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


target_train=y_train.tolist()
target_test=y_test.tolist()

In [33]:
smiles_list_train=train_set['smiles_canon'].tolist()
smiles_list_test=test_set['smiles_canon'].tolist()

In [34]:
# Prepare datasets
train_dataset = MolecularDataset(
    graph_data_list=[smiles_to_graph(smiles) for smiles in smiles_list_train], 
    physical_properties=X_train,#physical_properties_train, 
    targets=target_train)

test_dataset = MolecularDataset(
    graph_data_list=[smiles_to_graph(smiles) for smiles in smiles_list_test], 
    physical_properties=X_test,#physical_properties_test, 
    targets=target_test)


#train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=4, worker_init_fn=seed_worker)
#test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=4, worker_init_fn=seed_worker)



train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)  

In [35]:
# Define the Hybrid MPNN model class
class HybridMPNN(torch.nn.Module):
    def __init__(self, node_feature_dim, mpnn_hidden_dim, physical_property_dim, out_dim):
        super(HybridMPNN, self).__init__()

        # MPNN layers (Graph Convolutional Network)
        self.conv1 = GCNConv(node_feature_dim, mpnn_hidden_dim)
        self.conv2 = GCNConv(mpnn_hidden_dim, mpnn_hidden_dim)

        # Dense layer for physical properties
        self.physical_dense = nn.Linear(physical_property_dim, mpnn_hidden_dim)

        # Fully connected layers
        self.fc1 = nn.Linear(2 * mpnn_hidden_dim, mpnn_hidden_dim)
        self.fc2 = nn.Linear(mpnn_hidden_dim, out_dim)

    def forward(self, data, physical_properties):
        x, edge_index = data.x, data.edge_index

        # MPNN: Graph Conv layers
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # Global mean pooling over node embeddings
        x = global_mean_pool(x, data.batch)

        # Dense layer for physical properties
        physical_properties = F.relu(self.physical_dense(physical_properties))

        # Concatenate MPNN output with physical properties
        combined = torch.cat([x, physical_properties], dim=1)

        # Fully connected layers for prediction
        x = F.relu(self.fc1(combined))
        x = self.fc2(x)

        return x


In [36]:
def initialize_weights(model):
    for layer in model.modules():
        if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


In [38]:
import torch
from torch_geometric.data import DataLoader
from torch import nn
import torch.nn.functional as F

# Use nn.L1Loss for MAE
criterion = nn.L1Loss()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data, physical_properties, target in loader:
            data = data.to(device)
            physical_properties = physical_properties.to(device)
            target = target.to(device)

            out = model(data, physical_properties)
            loss = criterion(out.view(-1), target)
            total_loss += loss.item()
    return total_loss / len(loader)
# Training loop to track MAE
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    total_mae = 0  # To track MAE
    for data, physical_properties, target in loader:
        data = data.to(device)
        physical_properties = physical_properties.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        out = model(data, physical_properties)

        loss = criterion(out.view(-1), target)  # Use L1Loss (MAE)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

        # Calculate batch MAE
        batch_mae = torch.mean(torch.abs(out.view(-1) - target)).item()
        total_mae += batch_mae

    avg_loss = total_loss / len(loader)
    avg_mae = total_mae / len(loader)
    return avg_loss, avg_mae  # Return MAE along with loss

# Initialize the model, optimizer, and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


model = HybridMPNN(node_feature_dim=1, mpnn_hidden_dim=128, physical_property_dim=168, out_dim=1).to(device)
#optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
#optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # L2 regularization

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
initialize_weights(model)
import torch
import random

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0

# Initialize early stopping
early_stopping = EarlyStopping(patience=50, min_delta=0.005)

# Training loop with early stopping
def set_seed(seed=42):
    # Python random seed
    random.seed(seed)
    
    # Numpy random seed
    np.random.seed(seed)
    
    # PyTorch random seed for CPU
    torch.manual_seed(seed)
    
    # If using GPUs
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # If using multiple GPUs
    
    # Ensure deterministic behavior in GPU operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Call the seed setting function at the start of your script
set_seed(42)


num_epochs = 200  # Set a maximum number of epochs
for epoch in range(num_epochs):
    train_loss, train_mae = train(model, train_loader, optimizer, criterion)
    test_mae = evaluate(model, test_loader, criterion)  # Assume you have a validation set

    print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train MAE: {train_mae:.4f}, Test MAE: {test_mae:.4f}')

    # Call early stopping
    early_stopping(test_mae)

    # Check if we should stop early
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break

# After training,  evaluation of the model on the test data
final_test_mae = evaluate(model, test_loader, criterion)
print(f'Final Test MAE: {final_test_mae:.4f}')


Epoch 1/200, Train Loss: 0.8877, Train MAE: 0.8877, Test MAE: 0.5279
Epoch 2/200, Train Loss: 0.7381, Train MAE: 0.7381, Test MAE: 0.5791
Epoch 3/200, Train Loss: 0.6993, Train MAE: 0.6993, Test MAE: 0.4839
Epoch 4/200, Train Loss: 0.6688, Train MAE: 0.6688, Test MAE: 0.5170
Epoch 5/200, Train Loss: 0.6482, Train MAE: 0.6482, Test MAE: 0.4876
Epoch 6/200, Train Loss: 0.6328, Train MAE: 0.6328, Test MAE: 0.5113
Epoch 7/200, Train Loss: 0.6180, Train MAE: 0.6180, Test MAE: 0.5341
Epoch 8/200, Train Loss: 0.6051, Train MAE: 0.6051, Test MAE: 0.4702
Epoch 9/200, Train Loss: 0.5965, Train MAE: 0.5965, Test MAE: 0.4880
Epoch 10/200, Train Loss: 0.5868, Train MAE: 0.5868, Test MAE: 0.4803
Epoch 11/200, Train Loss: 0.5717, Train MAE: 0.5717, Test MAE: 0.5193
Epoch 12/200, Train Loss: 0.5684, Train MAE: 0.5684, Test MAE: 0.4983
Epoch 13/200, Train Loss: 0.5576, Train MAE: 0.5576, Test MAE: 0.4647
Epoch 14/200, Train Loss: 0.5506, Train MAE: 0.5506, Test MAE: 0.4577
Epoch 15/200, Train Loss: 0.5

In [41]:
# === Save the Model ===
torch.save(model.state_dict(), "../data/hybrid_mpnn_model_46.pth")
torch.save(optimizer.state_dict(), "../data/optimizer.pth")
print("Model and optimizer saved.")
node_feature_dim=1 
mpnn_hidden_dim=128 
physical_property_dim=168
out_dim=1
# === Reload the Model ===
loaded_model = HybridMPNN(node_feature_dim, mpnn_hidden_dim, physical_property_dim, out_dim).to(device)
loaded_model.load_state_dict(torch.load("../data/hybrid_mpnn_model_46.pth"))
loaded_model.eval()
print("Model reloaded.")

# Reload the optimizer (optional, if continuing training)
loaded_optimizer = torch.optim.Adam(loaded_model.parameters(), lr=0.001)
loaded_optimizer.load_state_dict(torch.load("../data/optimizer.pth"))

# === Evaluate the Reloaded Model ===
final_train_mae = evaluate(loaded_model, train_loader, criterion)
final_test_mae = evaluate(loaded_model, test_loader, criterion)
print(f"Reloaded Model - Final Train MAE: {final_train_mae:.4f}, Final Test MAE: {final_test_mae:.4f}")


Model and optimizer saved.
Model reloaded.
Reloaded Model - Final Train MAE: 0.3865, Final Test MAE: 0.5293


In [42]:
import math
from sklearn.metrics import r2_score

def evaluate1(model, loader):
    model.eval()
    total_squared_error = 0
    all_targets = []
    all_predictions = []
    n_samples = 0

    with torch.no_grad():
        for data, physical_properties, target in loader:
            data = data.to(device)
            physical_properties = physical_properties.to(device)
            target = target.to(device)

            # Model predictions
            out = model(data, physical_properties).view(-1)

            # Collect predictions and targets for R²
            all_predictions.extend(out.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

            # Compute squared errors
            squared_error = (out - target) ** 2
            total_squared_error += squared_error.sum().item()
            n_samples += target.size(0)

    # Compute RMSE
    rmse = math.sqrt(total_squared_error / n_samples)

    # Compute R² score using sklearn
    r2 = r2_score(all_targets, all_predictions)

    return rmse, r2


In [43]:
rmse, r2 = evaluate1(model, test_loader)
print(f'Test RMSE: {rmse:.4f}, Test R²: {r2:.4f}')

Test RMSE: 0.6653, Test R²: 0.8939
